## Download data of January 2025

In [41]:
from pathlib import Path
from urllib.request import urlretrieve


DATA_URL = (
    "https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2025-01.parquet"
)
DATA_DIR = Path("../data/raw")
PARQUET_PATH = DATA_DIR / "green_tripdata_2025-01.parquet"

DATA_DIR.mkdir(parents=True, exist_ok=True)

if PARQUET_PATH.exists():
    print(f"Reusing existing file: {PARQUET_PATH}")
else:
    print(f"Downloading {DATA_URL}")
    urlretrieve(DATA_URL, PARQUET_PATH)
    print(f"Saved file to {PARQUET_PATH}")

str(PARQUET_PATH)

Reusing existing file: ..\data\raw\green_tripdata_2025-01.parquet


'..\\data\\raw\\green_tripdata_2025-01.parquet'

## Exploring the data with Polars

In [42]:
import polars as pl

rides = pl.read_parquet(PARQUET_PATH)
print(f"Loaded {rides.height:,} rows and {rides.width} columns")
rides.schema

Loaded 48,326 rows and 21 columns


Schema([('VendorID', Int32),
        ('lpep_pickup_datetime', Datetime(time_unit='us', time_zone=None)),
        ('lpep_dropoff_datetime', Datetime(time_unit='us', time_zone=None)),
        ('store_and_fwd_flag', String),
        ('RatecodeID', Int64),
        ('PULocationID', Int32),
        ('DOLocationID', Int32),
        ('passenger_count', Int64),
        ('trip_distance', Float64),
        ('fare_amount', Float64),
        ('extra', Float64),
        ('mta_tax', Float64),
        ('tip_amount', Float64),
        ('tolls_amount', Float64),
        ('ehail_fee', Float64),
        ('improvement_surcharge', Float64),
        ('total_amount', Float64),
        ('payment_type', Int64),
        ('trip_type', Int64),
        ('congestion_surcharge', Float64),
        ('cbd_congestion_fee', Float64)])

In [43]:
rides[0]

VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,cbd_congestion_fee
i32,datetime[μs],datetime[μs],str,i64,i32,i32,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,i64,f64,f64
2,2025-01-01 00:03:01,2025-01-01 00:17:12,"""N""",1,75,235,1,5.93,24.7,1.0,0.5,6.8,0.0,null,1.0,34.0,1,1,0.0,0.0


In [44]:
rides.select(
    [
        "VendorID",
        "lpep_pickup_datetime",
        "lpep_dropoff_datetime",
        "payment_type",
        "total_amount",
    ]
).head()

VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,payment_type,total_amount
i32,datetime[μs],datetime[μs],i64,f64
2,2025-01-01 00:03:01,2025-01-01 00:17:12,1,34.0
2,2025-01-01 00:19:59,2025-01-01 00:25:52,2,11.1
2,2025-01-01 00:05:29,2025-01-01 00:07:21,2,26.55
2,2025-01-01 00:52:24,2025-01-01 01:07:52,1,36.77
2,2025-01-01 00:25:05,2025-01-01 01:01:10,1,46.86


In [45]:
daily_revenue = (
    rides
    .filter(pl.col("lpep_pickup_datetime").is_not_null())
    .group_by(
        pl.col("lpep_pickup_datetime").dt.date().alias("date")
    )
    .agg(
        pl.len().alias("rides"),
        pl.col("total_amount").sum().alias("daily revenue ($)")
    )
    .sort("date")
)

daily_revenue.head(10)

date,rides,daily revenue ($)
date,u32,f64
2024-12-25,1,43.2
2024-12-29,1,72.94
2024-12-31,4,64.03
2025-01-01,1001,25076.51
2025-01-02,1516,34256.83
2025-01-03,1525,36517.69
2025-01-04,1249,27869.2
2025-01-05,1129,25823.69
2025-01-06,1541,34049.13


## SQL queries with DuckDB (no need for setting up a database server)

In [46]:
from pathlib import Path

import duckdb

if not PARQUET_PATH.exists():
    raise FileNotFoundError("Download local data first!")

con = duckdb.connect()
con.sql("SELECT 1 AS ready").show()

┌───────┐
│ ready │
│ int32 │
├───────┤
│     1 │
└───────┘



In [47]:
daily_revenue_query = """
SELECT
    CAST(lpep_pickup_datetime AS DATE) AS date,
    COUNT(*) AS rides,
    SUM(total_amount) AS "daily revenue ($)"
FROM read_parquet(?)
WHERE lpep_pickup_datetime IS NOT NULL
GROUP BY CAST(lpep_pickup_datetime AS DATE)
ORDER BY date
"""

pl.from_arrow(con.sql(daily_revenue_query, params=[str(PARQUET_PATH)]).to_arrow_table()).head(10)

date,rides,daily revenue ($)
date,i64,f64
2024-12-25,1,43.2
2024-12-29,1,72.94
2024-12-31,4,64.03
2025-01-01,1001,25076.51
2025-01-02,1516,34256.83
2025-01-03,1525,36517.69
2025-01-04,1249,27869.2
2025-01-05,1129,25823.69
2025-01-06,1541,34049.13


In [48]:
another_monthly_query = """
SELECT
    CAST(lpep_pickup_datetime AS DATE) AS date,
    COUNT(*) AS rides,
    ROUND(AVG(total_amount), 2) AS avg_total_amount,
    ROUND(SUM(total_amount), 2) AS total_revenue
FROM read_parquet(?)
WHERE lpep_pickup_datetime IS NOT NULL
GROUP BY CAST(lpep_pickup_datetime AS DATE)
ORDER BY date
"""

pl.from_arrow(con.sql(another_monthly_query, params=[str(PARQUET_PATH)]).to_arrow_table()).head(10)

date,rides,avg_total_amount,total_revenue
date,i64,f64,f64
2024-12-25,1,43.2,43.2
2024-12-29,1,72.94,72.94
2024-12-31,4,16.01,64.03
2025-01-01,1001,25.05,25076.51
2025-01-02,1516,22.6,34256.83
2025-01-03,1525,23.95,36517.69
2025-01-04,1249,22.31,27869.2
2025-01-05,1129,22.87,25823.69
2025-01-06,1541,22.1,34049.13
